In [2]:
import json 
from tqdm import tqdm
from collections import Counter
from typing import List, Dict, Tuple

In [3]:
json_path = "/home/duypd/ThisPC-DuyPC/SG-Retrieval/Datasets/VisualGenome/Rev_v2.json"
with open(json_path) as f:
    data = json.load(f)

print(len(data))
print(data[0])

19444
{'qe': {'image_id': '2372880.jpg', 'trip': ['man wearing shoe', 'sidewalk near street', 'person wearing shirt', 'man riding skateboard', 'pant on man', 'man has head', 'man wearing jean', 'man has hair']}, 'rev': {'image_id': '2346677.jpg', 'image_id_qe': '2372880.jpg', 'trip': ['man wearing shoe', 'bag in car', 'person wearing shirt', 'man has head', 'man wearing jean', 'man wearing shirt', 'man has hair']}}


In [4]:

galleries = []
for item in tqdm(data):
    gallery_que = {}
    gallery_rev = {}
    gallery_que["image_id"] = item['qe']['image_id']
    gallery_que["trip"] = item['qe']['trip']

    gallery_rev["image_id"] = item['rev']['image_id']
    gallery_rev["trip"] = item['rev']['trip']

    galleries.append(gallery_que)
    galleries.append(gallery_rev)

    # break

len(galleries)

100%|██████████| 19444/19444 [00:00<00:00, 1089406.33it/s]


38888

In [5]:
with open('/home/duypd/ThisPC-DuyPC/SG-Retrieval/Datasets/VisualGenome/Galleries.json', 'w') as f:
    json.dump(galleries, f)

In [6]:
import os
from collections import defaultdict, Counter
from typing import List, Dict, Tuple

Dataset = List[Dict[str, List[str]]]

def normalize_triplet(t: str) -> str:
    return " ".join(t.lower().split())

def jaccard_set(query_trips: List[str], item_trips: List[str]) -> float:
    A = set(map(normalize_triplet, query_trips))
    B = set(map(normalize_triplet, item_trips))
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

def jaccard_multiset(query_trips: List[str], item_trips: List[str]) -> float:
    A = Counter(map(normalize_triplet, query_trips))
    B = Counter(map(normalize_triplet, item_trips))
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    inter = sum(min(A[t], B[t]) for t in set(A)|set(B))
    union = sum(max(A[t], B[t]) for t in set(A)|set(B))
    return inter / union

def norm_image_id(x) -> str:
    """So sánh id một cách robust: bỏ path, bỏ extension, ép chuỗi."""
    s = str(x)
    s = os.path.basename(s)       # imgs/2346401.jpg -> 2346401.jpg
    s = s.split("?")[0]           # phòng query có querystring
    s = os.path.splitext(s)[0]    # 2346401.jpg -> 2346401
    return s

def retrieve_by_jaccard(
    query_trips: List[str],
    db: Dataset,
    query_image_id: str,
    top_k: int = 10,
    use_multiset: bool = False,
    min_score: float = 0.2,
    dedup_mode: str = "max",   # hoặc "mean"
) -> Dict[str, object]:
    sim_fn = jaccard_multiset if use_multiset else jaccard_set
    qid_norm = query_image_id

    # 1) Tính score cho tất cả record
    scores_by_id = defaultdict(list)
    for item in db:
        # iid_norm = norm_image_id(item["image_id"])
        iid_norm = item["image_id"]
        if iid_norm == qid_norm:
            continue
        s = sim_fn(query_trips, item["trip"])
        scores_by_id[iid_norm].append(s)

    # 2) Gộp theo image_id
    if dedup_mode == "mean":
        best_by_id = {iid: sum(v)/len(v) for iid, v in scores_by_id.items() if v}
    else:
        best_by_id = {iid: max(v) for iid, v in scores_by_id.items() if v}

    # 3) Lọc theo min_score
    filtered = [(iid, s) for iid, s in best_by_id.items() if s >= min_score]

    # 4) Sort & Top-K
    ranked = sorted(filtered, key=lambda x: (-x[1], x[0]))[:top_k]

    # 5) Đóng gói thành dict duy nhất
    results = {
        "image_query": qid_norm,
        "trip_query": query_trips,
        "target": [iid for iid, _ in ranked],
        "score": [s for _, s in ranked]
    }
    return results



In [7]:
# query = [
#             "head of sheep",
#             "tree behind sheep",
#             "sheep has leg",
#             "sheep on hill",
#             "sheep has head"
#         ]
# query_image_id="2346401.jpg"
targets = []
for item in tqdm(galleries):
    query = item["trip"]
    query_image_id = item['image_id']
    tgt = retrieve_by_jaccard(query, galleries,
                            query_image_id=query_image_id,
                            top_k=5,
                            use_multiset=False,
                            min_score=0.2)
    targets.append(tgt)

len(targets)
with open('/home/duypd/ThisPC-DuyPC/SG-Retrieval/Datasets/VisualGenome/Target.json', 'w') as f:
    json.dump(targets, f) 
    # break

100%|██████████| 38888/38888 [2:25:32<00:00,  4.45it/s]  


In [1]:
len(targets)
with open('/home/duypd/ThisPC-DuyPC/SG-Retrieval/Datasets/VisualGenome/Target.json', 'w') as f:
    json.dump(targets, f)

NameError: name 'targets' is not defined

In [22]:
x = '2346677.jpg'
for i in targets:
    if(i['image_query'] == x):
        print(i)
        break

{'image_query': '2346677.jpg', 'trip_query': ['man wearing shoe', 'bag in car', 'person wearing shirt', 'man has head', 'man wearing jean', 'man wearing shirt', 'man has hair'], 'target': ['2406583.jpg', '2336255.jpg', '2338119.jpg', '2406755.jpg', '2318349.jpg'], 'score': [0.5714285714285714, 0.5555555555555556, 0.5555555555555556, 0.5555555555555556, 0.5]}
